# Suite FRM — The framework

Every dialect implements the same protocols (`mbse.Expressions.Framework`): its terms are values of the framework's
`Term`, the union of the dialects' `Term`s, stored through its install, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms` reads any dialect's terms by its kinds (`Terms.Kind`, `Terms.Dialect`); this suite
checks that machinery on every dialect at once, and on an extension of Patterns declared here. The dialects' own rules are
suite DIA, and translation suite TRN.

In [ ]:
import functools
import math

from mbse.Expressions import Framework, Translators
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Python3
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.Patterns.Expressions import Domains as BD, Evaluators as V, Expressions as E
from mbse.Expressions.C.Expressions import Evaluators as CV, Expressions as C
from mbse.Expressions.Cpp.Expressions import Evaluators as CPV, Expressions as CPP
from mbse.Expressions.Excel.Expressions import Evaluators as XV, Expressions as X
from mbse.Expressions.LaTeX.Expressions import Expressions as L
from mbse.Expressions.MATLAB.Expressions import Evaluators as MV, Expressions as M
from mbse.Expressions.Python.Python3 import Evaluators as PV, Expressions as P
from mbse.Expressions.SystemVerilog.Expressions import Evaluators as SVV, Expressions as SV
from mbse.Expressions.TypeScript.TypeScript5 import Evaluators as TV, Expressions as TS
from mbse.Expressions.Framework import Domains as D, Errors as FE, Evaluators as FV, Partials as FP, Symbolics as SY, \
    Terms as F, Translators as FT
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,  # the standard
                Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install,  # dialects first
                C.install, Cpp.install, SystemVerilog.install, TypeScript5.install]:
    store.transaction(install)
store.transaction(Patterns.define)  # Patterns' operations, which its validation reads
store.transaction(SystemVerilog.define)  # SystemVerilog's types, which its evaluation, inference and validation read
Term = store.registered("Expressions.Framework.Term")


def term(spec, dialect=E.DIALECT):
    """The term value `spec` writes, the root of a holder written in a transaction."""
    with store.transaction():
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(dialect.spec(spec))))


person = E.variable("person")
# TypeScript5 has no writers: its terms are written through the builders
ts_person = lambda u: u.identifier(lambda r: r.name("person"))  # noqa: E731: TypeScript5's `ts_person`, in any union that has identifiers
typescript5 = lambda u: u.sequenceexpression(lambda s: s.expressions([  # noqa: E731: every kind, written through the builders
    lambda e: e.parenthesizedexpression(lambda r: r.expression(lambda x: x.assignmentexpression(lambda a: a  # ({ a: [b, , ...c], d: { ts_person }, f = 1 } = ts_person)
        .left(lambda t: t.objectpattern(lambda o: o.properties([
            lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("a"))).value(
                lambda v: v.arraypattern(lambda ap: ap.elements([lambda y: y.identifier(lambda i: i.name("b")), lambda y: y.elision(lambda h: h),
                                                                  lambda y: y.restelement(lambda z: z.argument(lambda g: g.identifier(lambda i: i.name("c"))))])))),
            lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("d"))).value(
                lambda v: v.objectpattern(lambda op: op.properties([lambda y: y.property(lambda z: z.propertyKind("init").key(ts_person).shorthand(True).value(ts_person))])))),
            lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("f"))).shorthand(True).value(
                lambda v: v.assignmentpattern(lambda ap: ap.left(lambda g: g.identifier(lambda i: i.name("f"))).right(
                    lambda g: g.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)))))))])))
        .operator("=").right(ts_person)))),
    lambda e: e.functionexpression(lambda r: r.isAsync(True).generator(True).id(lambda i: i.name("g")).typeParameters(  # async function* g<T>(x: T = this.y) {}
        lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([lambda p: p.assignmentpattern(lambda ap: ap
            .left(lambda g: g.identifier(lambda i: i.name("x").typeAnnotation(lambda ta: ta.typeAnnotation(
                lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("T"))))))))
            .right(lambda g: g.memberexpression(lambda m: m.object(lambda o: o.thisexpression(lambda h: h)).property(lambda y: y.identifier(lambda i: i.name("y"))))))])),
    lambda e: e.arrowfunctionexpression(lambda r: r.isAsync(True).typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("U"))]))  # async <U>(p: U, ...r: U[]) => await import.meta.resolve(p)
        .params([lambda p: p.identifier(lambda i: i.name("p").typeAnnotation(lambda ta: ta.typeAnnotation(
                     lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("U"))))))),
                 lambda p: p.restelement(lambda z: z.argument(lambda g: g.identifier(lambda i: i.name("r"))).typeAnnotation(lambda ta: ta.typeAnnotation(
                     lambda t: t.tsarraytype(lambda at: at.elementType(lambda y: y.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("U")))))))))])
        .body(lambda b: b.awaitexpression(lambda w: w.argument(lambda c: c.callexpression(lambda ce: ce.callee(lambda f: f.memberexpression(lambda m: m
            .object(lambda o: o.metaproperty(lambda mp: mp.meta(lambda i: i.name("import")).property(lambda i: i.name("meta"))))
            .property(lambda y: y.identifier(lambda i: i.name("resolve"))))).arguments([lambda y: y.identifier(lambda i: i.name("p"))])))))),
    lambda e: e.classexpression(lambda r: r.decorators([lambda d: d.expression(ts_person)]).id(lambda i: i.name("K"))  # @ts_person class K<T> extends ts_person.Base<T> implements ts_person {}
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .superClass(lambda c: c.memberexpression(lambda m: m.object(ts_person).property(lambda y: y.identifier(lambda i: i.name("Base")))))
        .superTypeArguments(lambda a: a.params([lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("T"))))]))
        .implements([lambda c: c.expression(ts_person)])),
    lambda e: e.newexpression(lambda r: r.callee(lambda c: c.memberexpression(lambda m: m.object(ts_person).property(  # new ts_person.C<number>(...ts_person)
        lambda y: y.identifier(lambda i: i.name("C"))))).typeArguments(lambda a: a.params([lambda t: t.tsnumberkeyword(lambda k: k)]))
        .arguments([lambda y: y.spreadelement(lambda z: z.argument(ts_person))])),
    lambda e: e.taggedtemplateexpression(lambda r: r.tag(ts_person).quasi(lambda q: q.quasis([  # ts_person`a${ts_person}b`
        lambda y: y.raw("a"), lambda y: y.raw("b").tail(True)]).expressions([ts_person]))),
    lambda e: e.templateliteral(lambda r: r.quasis([lambda y: y.raw("x"), lambda y: y.raw("").tail(True)]).expressions([ts_person])),  # `x${ts_person}`
    lambda e: e.updateexpression(lambda r: r.operator("++").argument(ts_person)),  # ts_person++
    lambda e: e.unaryexpression(lambda r: r.operator("typeof").argument(ts_person)),  # typeof ts_person
    lambda e: e.binaryexpression(lambda r: r.left(lambda y: y.privateidentifier(lambda i: i.name("p"))).operator("in").right(ts_person)),  # #p in ts_person
    lambda e: e.logicalexpression(lambda r: r.left(ts_person).operator("??").right(lambda y: y.constant(lambda c: c.singleton("null")))),  # ts_person ?? null
    lambda e: e.conditionalexpression(lambda r: r.test(ts_person).consequent(  # ts_person ? /x/g : 1n
        lambda y: y.constant(lambda c: c.pattern("x").flags("g"))).alternate(lambda y: y.literal(lambda li: li.value(lambda n: n.as_scalar(1))))),
    lambda e: e.yieldexpression(lambda r: r.delegate(True).argument(ts_person)),  # yield* ts_person
    lambda e: e.tsasexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.tsstringkeyword(lambda k: k))),  # ts_person as string
    lambda e: e.tssatisfiesexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.tsobjectkeyword(lambda k: k))),  # ts_person satisfies object
    lambda e: e.tstypeassertion(lambda r: r.typeAnnotation(lambda t: t.tsanykeyword(lambda k: k)).expression(ts_person)),  # <any>ts_person
    lambda e: e.tsnonnullexpression(lambda r: r.expression(ts_person)),  # ts_person!
    lambda e: e.tsinstantiationexpression(lambda r: r.expression(ts_person).typeArguments(lambda a: a.params([lambda t: t.tsstringkeyword(lambda k: k)]))),  # ts_person<string>
    lambda e: e.arrayexpression(lambda r: r.elements([  # [1, , ...ts_person]: the framework's list, its elements any dialect's terms
        lambda y: y.typescript5(lambda t: t.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)))), lambda y: y.typescript5(lambda t: t.elision(lambda h: h)),
        lambda y: y.typescript5(lambda t: t.spreadelement(lambda z: z.argument(ts_person)))])),
    lambda e: e.objectexpression(lambda r: r.properties([  # { m(): void, ...ts_person, [ts_person]: "s" }
        lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("m"))).method(True).value(
            lambda v: v.tsemptybodyfunctionexpression(lambda f: f.returnType(lambda a: a.typeAnnotation(lambda t: t.tsvoidkeyword(lambda k: k)))))),
        lambda p: p.spreadelement(lambda z: z.argument(ts_person)),
        lambda p: p.property(lambda q: q.propertyKind("init").computed(True).key(ts_person).value(lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar("s")))))])),
    lambda e: e.jsxelement(lambda r: r  # <div id="x" {...ts_person}>{}{ts_person}<>text</></div>
        .openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i.name("div"))).attributes([
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxidentifier(lambda i: i.name("id"))).value(
                lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar("x")).text('"x"')))),
            lambda a: a.jsxspreadattribute(lambda z: z.argument(ts_person))]))
        .children([lambda c: c.jsxexpressioncontainer(lambda x: x.expression(lambda y: y.jsxemptyexpression(lambda h: h))),
                   lambda c: c.jsxexpressioncontainer(lambda x: x.expression(ts_person)),
                   lambda c: c.jsxfragment(lambda f: f.openingFragment(lambda h: h).children([lambda t: t.jsxtext(lambda j: j.raw("text"))]).closingFragment(lambda h: h))])
        .closingElement(lambda c: c.name(lambda n: n.jsxidentifier(lambda i: i.name("div"))))),
    lambda e: e.jsxfragment(lambda f: f.openingFragment(lambda h: h).children([lambda c: c.jsxelement(lambda x: x  # <><br /></>
        .openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i.name("br"))).selfClosing(True)))]).closingFragment(lambda h: h)),
    lambda e: e.chainexpression(lambda r: r.expression(lambda y: y.memberexpression(lambda m: m.object(ts_person).optional(True).property(  # ts_person?.a
        lambda z: z.identifier(lambda i: i.name("a")))))),
    lambda e: e.memberexpression(lambda m: m.object(lambda o: o.super(lambda h: h)).property(lambda y: y.identifier(lambda i: i.name("x")))),  # super.x
    lambda e: e.importexpression(lambda r: r.source(ts_person)),  # import(ts_person)
]))
c_person, cpp_person = C.identifier("person"), CPP.identifier("person")
c_int = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("int"))]))  # noqa: E731: `int`, in C's or C++'s
uint8 = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda n: n.name(  # noqa: E731: `uint8_t`, in C's or C++'s
    lambda m: m.identifier(lambda d: d.spelling("uint8_t"))))]))
c_every = lambda u: u.initializerlist(lambda r: r.items([  # noqa: E731: every kind of C's, written through the builders and its writers
    C.conditional(C.binary("&&", C.unary("!", C.member(c_person, "a")), C.binary(">", C.subscript(C.member(c_person, "b", "->"), 0), C.literal(1.5, "f"))),
                  C.cast(uint8, C.call("person", 1)), C.literal(5, "u")),  # the kinds C evaluates
    lambda i: i.constant(lambda c: c.singleton("nullptr")),  # nullptr
    lambda i: i.concatenatedstring(lambda c: c.parts([C.literal("a"), C.literal("b")])),  # "a" "b"
    lambda i: i.parenthesizedexpression(lambda p: p.expression(c_person)),  # (person)
    lambda i: i.genericselection(lambda g: g.controlling(c_person).associations([  # _Generic(person, int: 1, default: 0)
        lambda a: a.type(c_int).value(C.literal(1)), lambda a: a.value(C.literal(0))])),
    lambda i: i.postfixexpression(lambda p: p.operand(c_person).operator("++")),  # person++
    lambda i: i.sizeofexpression(lambda s: s.operand(c_int)),  # sizeof(int): a type name where an expression may stand
    lambda i: i.alignofexpression(lambda s: s.keyword("_Alignof").operand(c_person)),  # _Alignof(person)
    lambda i: i.assignmentexpression(lambda a: a.left(c_person).operator("+=").right(C.literal(1))),  # person += 1
    lambda i: i.designatedinitializer(lambda d: d.designators([  # .x[0] = person
        lambda x: x.fielddesignator(lambda f: f.name(lambda j: j.spelling("x"))), lambda x: x.indexdesignator(lambda f: f.index(C.literal(0)))]).initializer(
        lambda x: x.equalinitializer(lambda e: e.value(c_person)))),
    lambda i: i.compoundliteralexpression(lambda c: c.type(lambda t: t.typeid(lambda y: y.specifiers([  # (int[]){1}
        lambda s: s.primitivetypespecifier(lambda p: p.keyword("int"))]).declarator(lambda d: d.arraydeclarator(lambda a: a)))).initializer(
        lambda li: li.items([C.literal(1)]))),
    lambda i: i.statementexpression(lambda s: s),  # ({}): GNU's, its body of statements not part of the term
    lambda i: i.extensionexpression(lambda e: e.operand(c_person)),  # __extension__ person
]))
cpp_name = lambda spelling: lambda n: n.identifier(lambda i: i.spelling(spelling))  # noqa: E731: a C++ name that is an identifier
cpp_every = lambda u: u.initializerlist(lambda r: r.items([  # noqa: E731: every kind of C++'s, written through the builders and its writers
    CPP.conditional(CPP.binary("&&", CPP.unary("!", CPP.member(cpp_person, "a")), CPP.binary(">", CPP.subscript(CPP.member(cpp_person, "b", "->"), 0), CPP.literal(1.5, "f"))),
                    CPP.cast(uint8, CPP.call("person", 1)), CPP.literal(5, "u")),  # the kinds C++ evaluates
    lambda i: i.constant(lambda c: c.singleton("nullptr")),  # nullptr
    lambda i: i.userdefinedliteral(lambda d: d.literal(lambda li: li.value(lambda v: v.as_scalar(10))).suffix("_km")),  # 10_km
    lambda i: i.concatenatedstring(lambda c: c.parts([CPP.literal("a"), lambda x: x.literal(  # "a" R"(b)"
        lambda li: li.value(lambda v: v.as_scalar("b")).raw(True))])),
    lambda i: i.idexpression(lambda d: d.name(lambda n: n.qualifiedname(lambda q: q.qualifiers([cpp_name("std")]).name(cpp_name("cout"))))),  # std::cout
    lambda i: i.thisexpression(lambda t: t),  # this
    lambda i: i.parenthesizedexpression(lambda x: x.expression(cpp_person)),  # (person)
    lambda i: i.lambdaexpression(lambda la: la.captures([  # [=, &person](auto y) {}: its body of statements not part of the term
        lambda c: c.defaultcapture(lambda d: d.mode("=")), lambda c: c.simplecapture(lambda s: s.by_reference(True).name(lambda j: j.spelling("person")))]).declarator(
        lambda d: d.parameters([lambda q: q.parameterdeclaration(lambda pd: pd.specifiers([lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(
            lambda dd: dd.iddeclarator(lambda i2: i2.name(cpp_name("y")))))]))),
    lambda i: i.foldexpression(lambda f: f.left(cpp_person).operator("+")),  # (person + ...)
    lambda i: i.requiresexpression(lambda q: q.parameters([lambda a: a.parameterdeclaration(  # requires (int z) { z + 1; }
        lambda pd: pd.specifiers([lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]).declarator(
            lambda dd: dd.iddeclarator(lambda i2: i2.name(cpp_name("z")))))]).requirements([
        lambda r2: r2.simplerequirement(lambda s: s.expression(CPP.binary("+", CPP.identifier("z"), 1)))])),
    lambda i: i.functionalcastexpression(lambda f: f.type(lambda s: s.primitivetypespecifier(lambda q: q.keyword("int"))).initializer(  # int(person)
        lambda x: x.parenthesizedinitializer(lambda pi: pi.arguments([cpp_person])))),
    lambda i: i.postfixexpression(lambda x: x.operand(cpp_person).operator("--")),  # person--
    lambda i: i.namedcastexpression(lambda n: n.operator("static_cast").type(c_int).operand(cpp_person)),  # static_cast<int>(person)
    lambda i: i.typeidexpression(lambda t: t.operand(cpp_person)),  # typeid(person)
    lambda i: i.awaitexpression(lambda a: a.operand(cpp_person)),  # co_await person
    lambda i: i.sizeofexpression(lambda s: s.operand(c_int)),  # sizeof(int)
    lambda i: i.sizeofpackexpression(lambda s: s.pack(lambda j: j.spelling("Ts"))),  # sizeof...(Ts)
    lambda i: i.alignofexpression(lambda s: s.keyword("alignof").operand(c_int)),  # alignof(int)
    lambda i: i.noexceptexpression(lambda s: s.operand(cpp_person)),  # noexcept(person)
    lambda i: i.newexpression(lambda n: n.type(c_int).initializer(lambda x: x.initializerlist(lambda il: il.items([CPP.literal(1)])))),  # new int{1}
    lambda i: i.deleteexpression(lambda d: d.array(True).operand(cpp_person)),  # delete[] person
    lambda i: i.assignmentexpression(lambda a: a.left(cpp_person).operator("=").right(CPP.literal(0))),  # person = 0
    lambda i: i.throwexpression(lambda t: t),  # throw
    lambda i: i.yieldexpression(lambda y: y.operand(cpp_person)),  # co_yield person
    lambda i: i.packexpansion(lambda x: x.pattern(cpp_person)),  # person...
    lambda i: i.designatedinitializer(lambda d: d.designators([lambda x: x.fielddesignator(lambda f: f.name(lambda j: j.spelling("x")))]).initializer(  # .x = person
        lambda x: x.equalinitializer(lambda e: e.value(cpp_person)))),
    lambda i: i.statementexpression(lambda s: s),  # ({})
    lambda i: i.extensionexpression(lambda e: e.operand(cpp_person)),  # __extension__ person
]))
basic = E.let_("a", person.age, E.variable("a").ge(18).and_(person.has("email")))
samples = {  # one term per dialect, with every kind
    "Patterns": term(E.operation("and", basic, E.list_(1, E.mapping(("k", person.items)).item("k")).all(  # a quantifier too
        "x", E.variable("x").gt(0)))),
    "Python3": term(P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("person"), P.ifexp(  # every kind
        P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1, ("<", 2))),
                                P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2, 3), P.namedexpr("q", 0)),
        P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), P.tuple_(P.slice_(1, None, -1), P.constant(...))),
               P.starred(P.list_(1)), P.keyword("k", P.dict_(("k", 2), (None, P.name("a"))))),
        P.call("f", P.generatorexp(P.subscript(P.name("p"), 0), P.comprehension("p", P.attribute(P.name("a"), "z"), (P.name("p"),))),
               P.listcomp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("r"))), P.name("a"), is_async=True)),
               P.setcomp(P.name("p"), P.comprehension("p", P.name("a"))), P.dictcomp(P.name("p"), None, P.comprehension("p", P.name("a"))),
               P.set_(P.constant(None), P.constant(1j)), P.lambda_(P.name("b"), (P.arg("b", 1),)), P.yield_(), P.yieldfrom(P.name("a")),
               P.await_(P.name("a")), P.concatenatedstring("x", P.joinedstr(("y", P.formattedvalue(P.name("a"), "r")))),
               P.templatestr((P.interpolation(P.name("a")),)), P.parenthesized(P.name("a"))))),
        alias="f"), alias="np"), P.DIALECT),
    "MATLAB": term(M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("person"), "age")),
                                               M.call("isfield", "x", M.index(M.array(M.arrayfun("p", M.identifier("person"), M.identifier("p")), 2), 1)))), M.DIALECT),
    "Excel": term(X.let_("a", X.field(X.name("person"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
                                                                              X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2)))))), X.DIALECT),
    "LaTeX": term(L.where("a", L.member(L.symbol("person"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18), L.binary(
        "\\lor", L.unary("\\lnot", L.function("has", L.symbol("person"), "email")), L.binary(">", L.frac(L.symbol("a"), 2), 1.5)))), L.DIALECT),
    "C": term(c_every, C.DIALECT),
    "Cpp": term(cpp_every, CPP.DIALECT),
    "SystemVerilog": term(SV.binary("->", SV.inside(SV.member(SV.identifier("person"), "a"), 1, SV.span(2, 5)), SV.conditional(  # every kind
        SV.unary("!", SV.select(SV.identifier("person"), 0)), SV.concatenation(SV.replication(2, SV.vector("10")), SV.range_(SV.identifier("person"), 3, 0)),
        SV.cast("byte", SV.call("$clog2", SV.iterate(SV.method(SV.identifier("person"), "unique"), "sum", "p", SV.identifier("p")))))), SV.DIALECT),
    "TypeScript5": term(typescript5, TS.DIALECT),
}
DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, CPP.DIALECT, SV.DIALECT, TS.DIALECT]
EVALUATORS = {E.DIALECT: V.OfAny, P.DIALECT: PV.OfAny, M.DIALECT: MV.OfAny, X.DIALECT: XV.OfAny,
              C.DIALECT: CV.OfAny, CPP.DIALECT: CPV.OfAny, SV.DIALECT: SVV.OfAny, TS.DIALECT: TV.OfAny}  # LaTeX has none
INVALID = {"error", "missing"}  # the framework's kinds of invalid syntax, which every dialect's `Term` holds

<a id="FRM-01"></a>
## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
assert all(isinstance(evaluate, FV.Evaluator) for evaluate in EVALUATORS.values())
for dialect in DIALECTS:
    assert isinstance(dialect, F.Dialect) and all(isinstance(kind, F.Kind) for kind in dialect.kinds().values())
    for node in F.walk(samples[dialect.name()]):  # a term of its own union, nested in its kinds or in shared ones
        assert F.is_term(node) and F.dialect_of(node) == dialect.union and F.kind_of(node)[0] in dialect.kinds()
assert [d.name() for d in DIALECTS] == ["Patterns", "Python3", "MATLAB", "Excel", "LaTeX", "C", "Cpp", "SystemVerilog",
                                     "TypeScript5"]
assert repr(E.DIALECT) == "<dialect Patterns>" and [d.tag for d in DIALECTS] == [b.name for b in Term.branches]
assert [d.union for d in DIALECTS] == [b.schema.name for b in Term.branches]  # the union of the dialects' `Term`s
# Every dialect a branch of the framework's kind union of dialects: the standard dialects first, which have no terms;
# the framework's union of kinds is of the dialects' unions, each holding its dialect's kinds.
Dialects, DomainKind = store.registered("Expressions.Framework.Dialect"), store.registered("Expressions.Framework.DomainKind")
assert [b.name for b in Dialects.branches] == ["isoiec11404", "ieee754", "ieee1164", "unicode", *[d.tag for d in DIALECTS]]
assert [F.dialect_name(b.schema.singleton.removeprefix("Expressions.")) for b in Dialects.branches[:4]] == [
    "ISOIEC11404", "IEEE754", "IEEE1164", "Unicode"]
assert {b.name: [k.name for k in b.schema.branches] for b in DomainKind.branches} == {
    "isoiec11404": ["boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"], "ieee754": ["ieee754"],
    "ieee1164": ["ieee1164"], "unicode": ["unicode"], "python3": ["str", "nonetype", "ellipsis"], "matlab": ["string"],
    "excel": ["text", "error"],
    "c": ["nullptr_t"], "cpp": ["nullptr_t"], "systemverilog": ["logic", "bit"], "typescript5": ["string", "null"]}
assert [b.schema.name for b in DomainKind.branches[4:]] == [  # each dialect's own union of kinds, named by its path
    d.union.removesuffix(".Term") + ".DomainKind" for d in DIALECTS if d.name() not in ("Patterns", "LaTeX")]
assert all(list(b.schema.adjacencies) == ["domains", "operations"] for b in Dialects.branches)
assert list(X.DIALECT.kinds()) == ["literal", "variable", "cell", "let", "call", "infix", "prefix", "field", "map", "error",
                                   "missing"]
for dialect in DIALECTS:  # each union's branches are its dialect's kinds, named plainly; shared kinds, the framework's
    union = store.registered(dialect.union)
    assert {b.name for b in union.branches} == set(dialect.kinds()), dialect.name()
    shared = {b.schema.name.rsplit(".", 1)[1].lower() for b in union.branches if b.schema.name.rpartition(".")[0] == "Expressions.Framework"}
    assert shared <= {"literal", "call", "variable", "let", "list", "mapping", "error", "missing"} and INVALID <= shared
C_UNION, CPP_UNION = store.registered(C.DIALECT.union), store.registered(CPP.DIALECT.union)
assert [b.name for b in C_UNION.branches] == list(C.DIALECT.kinds()) and [b.name for b in CPP_UNION.branches] == list(CPP.DIALECT.kinds())
assert {b.name for b in C_UNION.branches} - {b.name for b in CPP_UNION.branches} == {"variable", "genericselection", "compoundliteralexpression"}
assert "variable" not in CPP.DIALECT.kinds()  # C++ writes a name only as an idexpression, its grammar's one kind
assert {b.name for b in CPP_UNION.branches} - {b.name for b in C_UNION.branches} == {  # C++ is not a superset of C
    "userdefinedliteral", "idexpression", "thisexpression", "lambdaexpression", "foldexpression", "requiresexpression",
    "functionalcastexpression", "namedcastexpression", "typeidexpression", "awaitexpression", "sizeofpackexpression",
    "noexceptexpression", "newexpression", "deleteexpression", "throwexpression", "yieldexpression", "packexpansion"}
assert all(b.schema.name.startswith("Expressions.C.Expressions.") for b in C_UNION.branches if b.name not in ("variable", "error", "missing"))
assert all(b.schema.name.startswith("Expressions.Cpp.Expressions.") for b in CPP_UNION.branches if b.name not in ("error", "missing"))
assert list(E.DIALECT.kinds()) == ["literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]
assert [kind.role for kind in E.DIALECT.kinds().values()] == [
    F.LITERAL, F.APPLICATION, F.REFERENCE, F.BINDING, F.APPLICATION, F.APPLICATION, F.QUANTIFIER, F.INVALID, F.INVALID]
for domain in [D.Anything, D.OfTypes("int", int), D.OfValues("odd", lambda v: v % 2 == 1), D.OfUnion(D.Anything)]:
    assert isinstance(domain, D.Domain)
for signature in [D.Function((), D.Anything), D.Same(1, (D.Anything,)), D.Overloaded(D.Function((), D.Anything))]:
    assert isinstance(signature, D.Signature)
for translator in Translators.TRANSLATORS:
    assert isinstance(translator, FT.Translator) and isinstance(translator.inverse(), FT.Pairwise)
# The package exports the framework alone; each dialect, Patterns among them, is imported from its own path.
import mbse.Expressions as package
assert package.__all__ == ["Framework"] and package.Framework is Framework
assert not any(hasattr(package, name) for name in ("Domains", "Evaluators", "Expressions", "Partials"))

<a id="FRM-02"></a>
## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
for dialect in DIALECTS:
    kinds = set()
    for node in F.walk(samples[dialect.name()]):
        form = F.form(node)
        kinds.add(form.kind)
        made = term(dialect.make(form))
        assert F.same(made, node) and F.form(made).kind == form.kind and made is not node
    assert kinds == set(dialect.kinds()) - INVALID, (dialect.name(), kinds)  # the samples have every valid kind
assert F.form(term(3)) == F.Form("literal", {"value": 3})
assert F.form(term(E.let_("x", None, None))) == F.Form("let", {"name": "x"}, (None, None))  # empty slots are None
assert F.form(term(lambda u: u.call(lambda c: c))) == F.Form("call")
with raises(ValueError, match="Patterns has no kind 'constant'"):
    E.DIALECT.make(F.Form("constant"))
with raises(ValueError, match="a literal has no attribute 'name'"):
    term(E.DIALECT.make(F.Form("literal", {"name": "x"})))
with raises(ValueError, match="a let takes 2 arguments, got 1"):
    term(E.DIALECT.make(F.Form("let", {"name": "x"}, (basic,))))
with raises(ValueError, match="a quantifier takes 2 arguments, got 3"):
    term(E.DIALECT.make(F.Form("quantifier", {"name": "x", "quantifier": "all"}, (1, 2, 3))))
spelled = term(lambda u: u.literal(lambda r: r.text("08").problems(["leading zeros"])))  # a container of scalars too
assert F.same(term(E.DIALECT.make(F.form(spelled))), spelled)
made = term(E.DIALECT.make(F.Form("call", {"callable": "f"}, (E.variable("g"), 1, 2))))  # specs: writers, natives
assert [F.kind_of(a)[0] for a in F.form(made).arguments] == ["variable", "literal", "literal"]
with raises(ValueError, match="a term holds no dialect"):
    F.kind_of(term(lambda u: u))  # nothing written: no branch of Term is selected
with store.transaction():  # a value of a dialect's `Term`, alone
    alone = store.Expressions.Patterns.Expressions.Term(lambda u: u)
assert F.is_term(alone) and F.dialect_of(alone) == E.DIALECT.union
with raises(ValueError, match="a term holds no kind"):
    F.kind_of(alone)
with raises(ValueError, match="a call takes at least 1 arguments, got 0"):  # its function, then its arguments
    term(P.DIALECT.make(F.Form(P.CALL)), P.DIALECT)
made = term(P.DIALECT.make(F.Form(P.CALL, {}, (P.name("f"), P.constant(1), 2))), P.DIALECT)  # a native: Python's constant
assert F.name_of(F.form(made).arguments[0]) == "f" and [F.form(a) for a in F.form(made).arguments[1:]] == [
    F.Form(P.LITERAL, {"value": 1}), F.Form(P.LITERAL, {"value": 2})]
assert F.kind_of(term(1, M.DIALECT))[0] == "literal" and F.kind_of(term(True, C.DIALECT))[0] == "literal"  # its literal kind
with raises(TypeError, match="expected a term, a writer or a native value, got [1]"):  # no literal kind holds these
    E.DIALECT.spec([1])
with raises(TypeError, match="an Excel literal cannot hold a bytes"):
    X.DIALECT.spec(b"x")
# A form and a part print as Python writes them, in messages (both languages alike)
assert repr(F.Form("call", {"callable": "add"}, ("x",))) == "Form(kind='call', attributes={'callable': 'add'}, arguments=('x',))"
assert repr(F.Part("keyword", {"arg": "x"}, 1)) == "Part(kind='keyword', attributes={'arg': 'x'}, size=1)"

<a id="FRM-03"></a>
## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
shared = E.variable("x")
pair = term(E.operation("f", shared, shared))
assert [F.form(e).kind for e in F.walk(pair)] == ["call", "variable", "variable"]  # a nested value: each visited
assert [F.form(e).attributes for e in F.walk(term(basic))][:3] == [{"name": "a"}, {"callable": "get"}, {"name": "person"}]
assert F.fold(pair, lambda node, results: 1 + sum(results)) == 3
calls = []
F.fold(pair, lambda node, results: calls.append(node))
assert len(calls) == 3  # once per term
assert F.fold(term(E.let_("x", None, 1)), lambda node, results: results) == [None, []]
for a, b, expected in [(1, 1, True), (1, 1.0, False), (math.nan, math.nan, True), (-0.0, 0.0, False),
                       (0.0, 0.0, True), ("x", "x", True), (b"x", b"x", True), (b"x", b"y", False)]:
    assert F.same(term(a), term(b)) is expected, (a, b)
assert F.same(term(basic), term(E.DIALECT.make(F.form(term(basic))))) and F.same(term(E.let_("x", None, None)), term(E.let_("x", None, None)))
assert not F.same(term(E.let_("x", None, None)), term(E.let_("x", 1, 2))) and not F.same(term(1), term(E.variable("x")))
assert not F.same(term(E.operation("f", 1)), term(E.operation("f", 1, 2)))
assert not F.same(term(E.operation("f")), term(lambda u: u.call(lambda c: c)))  # attributes differ

<a id="FRM-04"></a>
## FRM-04 · Every dialect's install adds its union of its kinds to `Term`, and its terms round-trip

In [ ]:
for dialect in DIALECTS:
    assert dialect.tag in {b.name for b in Term.branches}
    sample = samples[dialect.name()]
    for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
        back = decode(Term, encode(Term, sample))
        assert F.same(back, sample) and dialect.validate(term(back, dialect), bound={"person"}) == []  # held, as validation reads its store
# The framework reads a kind's slots, variadic arguments and attributes from its schema: a slot holds Term, or a
# dialect's union.
of = {d.name(): {b.name: b.schema for b in store.registered(d.union).branches} for d in DIALECTS}
kinds = of["Patterns"]
assert F._parts(kinds["let"]) == (("value", "body"), None, ("name",))
assert F._parts(kinds["call"]) == ((), "arguments", ("callable",))
assert F._parts(kinds["quantifier"]) == (("collection", "body"), None, ("name", "quantifier"))
assert F._parts(kinds["literal"]) == ((), None, ("value", "text", "problems"))
assert F._parts(kinds["list"]) == ((), "elements", ()) and F._parts(kinds["mapping"]) == ((), "entries", ())
assert F._parts(kinds["error"]) == ((), "terms", ("text",)) and F._parts(kinds["missing"]) == ((), None, ("kind",))
assert F._members(kinds["mapping"]) == ("key", "value") and F._members(kinds["list"]) == F._members(kinds["let"]) == ()
for dialect in DIALECTS[1:]:
    for tag in dialect.kinds():  # a dialect's own kinds are its schemas, named by it
        assert of[dialect.name()][tag].name.startswith((f"Expressions.{dialect.path}.", "Expressions.Framework.")), tag
assert [of["Python3"][tag].name for tag in P.DIALECT.kinds()][-10:] == [
    "Expressions.Python.Python3.Compare", "Expressions.Python.Python3.BoolOp", "Expressions.Python.Python3.NamedExpr", "Expressions.Python.Python3.IfExp",
    "Expressions.Python.Python3.Lambda", "Expressions.Framework.Let", "Expressions.Python.Python3.Import", "Expressions.Python.Python3.ImportFrom",
    "Expressions.Framework.Error", "Expressions.Framework.Missing"]
assert of["Python3"][P.LITERAL].refines is store.registered("Expressions.Framework.Literal")  # the framework's, refined
assert F._parts(of["Python3"][P.CALL]) == (("func",), "args", ("keywords",))  # a slot, variadic arguments, then helpers
assert F._helpers(of["Python3"][P.CALL]) == ("keywords",) and F._helpers(of["Python3"][P.BINOP]) == ()
assert F._parts(of["Python3"][P.GENERATOREXP]) == (("elt",), None, ("generators",))
assert F._parts(of["Python3"][P.DICT]) == ((), "items", ()) and F._members(of["Python3"][P.DICT]) == ("key", "value")
assert F._parts(of["MATLAB"]["binary"]) == (("left", "right"), None, ("operator",))
assert F._parts(of["Excel"]["cell"]) == ((), None, ("address", "sheet", "book"))
assert of["Python3"][P.CALL].properties["func"].schema is store.registered(P.DIALECT.union)  # its own union
assert of["C"]["binaryexpression"].properties["left"].schema is store.registered(C.DIALECT.union)  # C's kinds hold C's terms
assert of["Cpp"]["binaryexpression"].properties["left"].schema is store.registered(CPP.DIALECT.union)  # and C++'s, C++'s
# Each dialect's holder, the root of an expression tree: a reference object refining the framework's Expression, its
# root narrowed to the dialect's Term. It validates, holds a term of its dialect, round-trips, prints (FRM-11), and HasRoot,
# which links the framework's Expression, links it.
Expression = store.registered("Expressions.Framework.Expression")
holders = {}
for dialect in DIALECTS:
    Holder = store.registered(f"Expressions.{dialect.path}.Expression")
    assert Holder.ref and Holder.refines is Expression and Holder.validate() == [] and F.is_expression(Holder)
    assert Holder.properties["root"].schema is store.registered(dialect.union)  # narrowed from Expression's Term
    sample = samples[dialect.name()]
    with store.transaction():
        held = functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression(lambda e: e.root(dialect.spec(sample)))
        linked = store.Schemas.OfExpression(lambda s: s.name(f"frm.{dialect.name()}").expression(
            lambda a: a.insert(lambda e: e.expression(held))))
    holders[dialect] = held
    assert F.same(F.term(held), sample) and F.dialect_of(F.term(held)) == dialect.union
    assert Validators.Validate(store)(Holder, held) == [] and dialect.validate(held, bound={"person"}) == []
    assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, held)).root, held.root)
    assert [F.same(entry.expression, held) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
with raises(Exception, match="'quantifier' is not a member of a union value"):  # its root holds its dialect's terms alone
    with store.transaction():
        store.Expressions.Python.Python3.Expression(lambda e: e.root(E.list_(1).all("p", True)))

<a id="FRM-05"></a>
## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
assert [d.validate(samples[d.name()], bound={"person"}, core=True) for d in DIALECTS] == [[]] * len(DIALECTS)
assert E.DIALECT.validate(term(E.operation("add", 1))) == ["add takes 2 arguments, got 1"]
assert E.DIALECT.validate(term(E.operation("not", E.variable("x")))) == ["argument 0: variable 'x' is not bound"]
assert E.DIALECT.validate(term(E.mapping((E.variable("k"), 1), ("b", E.list_(E.variable("x")))))) == [
    "key 0: variable 'k' is not bound", "value 1: argument 0: variable 'x' is not bound"]  # a mapping's terms, by entry
assert E.DIALECT.validate(term(lambda u: u.mapping(lambda m: m.entries([lambda a: a.as_object(lambda o: o.key(E.literal(1)))])))) == [
    "a mapping needs a value 0"]
assert E.DIALECT.validate(term(E.let_("a", E.variable("a"), E.variable("a")))) == ["value: variable 'a' is not bound"]
assert E.DIALECT.validate(term(lambda u: u.let(lambda r: r.name("x").value(E.literal(1))))) == ["a let needs a body"]
for spec, bound, problems in [
    (P.subscript(P.name("t"), "x"), (), ["value: variable 't' is not bound"]),
    (P.call("f", 1), (), ["func: variable 'f' is not bound"]),  # a slot, then variadic arguments
    (P.call(P.name("f"), P.name("x")), {"f"}, ["argument 0: variable 'x' is not bound"]),
    (P.call(P.name("f"), P.keyword("k", P.name("x"))), {"f"}, ["keywords 0 value: variable 'x' is not bound"]),  # a helper's
    (lambda u: getattr(u, P.CALL)(lambda r: r.func(P.name("f")).keywords([lambda a: a.as_object(lambda o: o.arg("k"))])),
     {"f"}, ["a call needs a keywords 0 value"]),
    (P.listcomp(P.name("y"), P.comprehension("y", P.name("y"))), (), ["generators 0 iter: variable 'y' is not bound"]),
    (P.lambda_(P.name("a"), ("a", P.arg("b", P.name("a")))), (), ["args args 1 default_value: variable 'a' is not bound"]),
    (P.subscript(P.name("t"), P.slice_()), {"t"}, []),  # a kind's optional slots may be empty
    (P.import_("numpy", P.name("np"), alias="np"), (), []),  # an import binds a name; its alias is optional
    (P.import_("numpy", P.name("np")), (), ["body: variable 'np' is not bound"]),
    (lambda u: getattr(u, P.IMPORT)(lambda r: r.module("numpy")), (), ["an import needs a body"]),
    (P.name("not a name"), (), ["a name must be an identifier, got 'not a name'"]),  # a kind's own check
]:
    assert P.DIALECT.validate(term(spec, P.DIALECT), bound) == problems
assert X.DIALECT.validate(term(X.cell("A1"), X.DIALECT)) == [] and X.DIALECT.validate(term(X.cell("B2", "Sheet1"), X.DIALECT), core=True) == []  # cells need no binding
assert M.DIALECT.validate(term(lambda u: u.field(lambda r: r), M.DIALECT)) == ["a field needs a name", "a field needs a value"]
assert M.DIALECT.validate(term(lambda u: u.binary(lambda r: r.operator("&&").left(M.constant(1))), M.DIALECT)) == ["a binary needs a right"]
assert M.DIALECT.validate(term(M.binary("**", 1, 2), M.DIALECT)) == [] and M.DIALECT.validate(term(M.binary("**", 1, 2), M.DIALECT), core=True) == [
    "'**' is not a core operation"]
assert M.DIALECT.validate(term(M.field(M.identifier("s"), "any name"), M.DIALECT), {"s"}, core=True) == []  # a field's vocabulary is open
assert X.DIALECT.validate(term(X.function("NOT", True, False), X.DIALECT)) == ["NOT takes 1 arguments, got 2"]
assert X.DIALECT.validate(term(X.function("AND", True), X.DIALECT)) == []  # AND: any number
assert X.DIALECT.validate(term(X.let_("a", X.name("a"), X.name("a")), X.DIALECT)) == ["value: variable 'a' is not bound"]
assert X.DIALECT.validate(term(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(b"x"))), X.DIALECT)) == [
    "an Excel literal cannot hold a bytes"]  # the dialect's own check of the framework's literal
assert M.DIALECT.validate(term(lambda u: u.call(lambda r: r.callable(7)), M.DIALECT)) == ["a call's callable must be a str, got int"]

<a id="FRM-06"></a>
## FRM-06 · Domains and signatures

In [ ]:
Int, Float, Str, Odd = D.OfTypes("int", int), D.OfTypes("float", float), D.OfTypes("str", str), D.OfValues("odd", lambda v: v % 2 == 1)
Number = D.OfUnion(Int, Float)
assert Int.contains(1) and not Int.contains(True) and not Int.contains(1.0) and repr(Int) == "int"
assert D.OfTypes("bool", bool).contains(True) and not D.OfTypes("bool", bool).contains(1)
assert Str.contains("a") and not Str.contains(b"a")
assert Number.name() == "int | float" and repr(Number) == "int | float" and Number.contains(1.5) and not Number.contains("1")
assert Number.includes(Int) and Number.includes(D.OfUnion(Float, Int)) and not Int.includes(Number)
assert D.OfTypes("num", int, float).includes(Number) and not Number.includes(Str) and not Number.includes(D.Anything)
assert Odd.contains(3) and Odd.includes(Odd) and not Odd.includes(Int) and Odd.includes(D.OfUnion(Odd)) and repr(Odd) == "odd"
assert D.Anything.includes(Number) and D.Anything.contains(None) and repr(D.Anything) == "any" and D.Anything.name() == "any"
assert D.overlaps(Int, D.Anything) and D.overlaps(Number, Int) and not D.overlaps(Int, Str)
assert D.OfTypes("list", list).contains([1]) and not D.OfTypes("list", list).contains("x")  # a class, by instance
f = D.Function((Int, Str), Float)
assert f.arity() == 2 and f.result([Int, Str]) is Float and f.result([D.Anything, Str]) is Float
assert f.result([Str, Str]) is None and f.result([Int]) is None and f.describe() == "(int, str) -> float"
assert f.exact([Int, Str]) and not f.exact([D.Anything, Str])
same = D.Same(2, (Int, Float))
assert same.result([Int, Int]) is Int and same.result([Int, Float]) is None and same.result([Int]) is None
assert same.result([D.Anything, D.Anything]).name() == "int | float" and same.result([D.Anything, Float]) is Float
assert same.describe() == "(T, T) -> T for T in int | float" and D.Same(1, (Int,), Str).describe() == "(T) -> str for T in int"
assert D.Same(2, (Int, Float), Str).result([D.Anything, D.Anything]) is Str  # one result, however many match
assert D.Opaque().arity() == -1 and D.Opaque(Int).result([Str, Str, Str]) is Int and D.Opaque().describe() == "(...) -> any"
either = D.Either(2)
assert either.arity() == 2 and either.result([Int, Int]) is Int and either.result([Int, Str]).name() == "int | str"
assert either.result([Int]) is None and either.describe() == "(T, T) -> T"
any_number = D.Either(-1)  # Python's `and` of any number of values
assert any_number.result([Int, Str, Int]).name() == "int | str" and any_number.result([]) is None and any_number.describe() == "(T, ...) -> T"
assert same.exact([Int, Int]) and not same.exact([Int, Float]) and not same.exact([Int])
over = D.Overloaded(D.Same(2, (Int,)), D.Function((Number, Number), Float))
assert over.arity() == 2 and over.result([Int, Int]) is Int and over.result([Int, Float]) is Float
assert over.result([D.Anything, Int]).name() == "int | float" and over.result([Str, Str]) is None
assert over.describe() == "(T, T) -> T for T in int | (int | float, int | float) -> float"

<a id="FRM-07"></a>
## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:


def infer(spec, environment=None):
    return E.DIALECT.infer(term(spec), environment)


assert infer(basic, {"person": BD.Object}) is BD.Bool
assert infer(E.literal(1).add(2)) is BD.Int and infer(E.literal(1.5).neg()) is BD.Float
assert infer(person.age, {"person": BD.Object}) is D.Anything
assert infer(person.age.add(1), {"person": BD.Object}) is BD.Int  # only int + int applies
assert M.DIALECT.infer(term(M.call("dist", 1, 2), M.DIALECT)) is D.Anything  # an extension: nothing is known of it
with raises(ValueError, match="cannot infer the domain of an invalid expression: 'contains' is not an operation Patterns defines"):
    infer(E.operation("contains", 1, 2))  # Patterns' operations are the ones it defines
assert infer(E.list_(1, 2)).name() == "list(int)" and infer(E.list_(1, "a")).name() == "list(int | str)"
assert infer(E.list_()).name() == "list(any)" and infer(E.list_(1, 2).count()) is BD.Int
assert infer(E.mapping(("a", 1.5))).name() == "keyed(str, float)" and infer(E.mapping()).name() == "keyed(any, any)"
assert infer(E.list_(1, 2).item(0)) is BD.Int and infer(E.list_(1, 2).all("x", E.variable("x").gt(0))) is BD.Bool
assert BD.LIST.describe() == "(T, ...) -> list(T)" and BD.MAPPING.describe() == "(K, T, ...) -> keyed(K, T)" and BD.LIST.arity() == -1
assert infer(E.let_("a", 1, E.let_("b", "s", E.variable("a")))) is BD.Int
with raises(TypeError, match="add cannot take (int, float); it takes (T, T) -> T for T in int | float"):
    infer(E.literal(1).add(1.5))
with raises(TypeError, match="has cannot take (int, str); it takes (object | record, str) -> bool"):
    infer(E.literal(1).has("x"))
with raises(ValueError, match="cannot infer the domain of an invalid expression: variable 'person' is not bound"):
    infer(person)
assert BD.of(term(1)) is BD.Object and BD.of(b"") is BD.Bytes  # a term value is an object
with raises(TypeError, match="a list is not a value of the Patterns dialect"):
    BD.of([])
assert M.DIALECT.infer(samples["MATLAB"], {"person": D.Anything}).name() == "logical"
assert X.DIALECT.infer(samples["Excel"], {"person": D.Anything}).name() == "logical"
assert L.DIALECT.infer(samples["LaTeX"], {"person": D.Anything}).name() == "truth"
assert P.DIALECT.infer(samples["Python3"], {"person": D.Anything}) is D.Anything  # a call gives what is not known
assert TS.DIALECT.infer(samples["TypeScript5"], {"person": D.Anything}) is D.Anything  # nothing is known but a literal's
assert P.DIALECT.infer(term(P.boolop("and", 1, "a"), P.DIALECT)).name() == "int | str"  # Python's and gives an operand
assert P.DIALECT.infer(term(P.import_("math", P.binop("+", 1, True)), P.DIALECT)).name() == "int"

<a id="FRM-08"></a>
## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
interpreter = FV.Interpreter(E.DIALECT, {"call": {}, "quantifier": {}})
with raises(TypeError, match="not a term: 3"):
    interpreter(3)
with raises(ValueError, match="a literal needs a value"):
    interpreter(term(E.literal(None)))
with raises(KeyError, match="variable 'x' is not bound"):
    interpreter(term(E.variable("x")))
with raises(NotImplementedError, match="'and' is not an operation Patterns defines"):
    interpreter(term(E.literal(True).and_(False)))
with raises(NotImplementedError, match="this evaluator has no values of unsigned int"):  # a kind's own domain: C's type
    FV.Interpreter(C.DIALECT, {})(term(C.literal(5, "u"), C.DIALECT))
with raises(TypeError, match="not takes 1 arguments, got 2"):
    FV.Interpreter(E.DIALECT, {"call": {"not": None}})(term(E.operation("not", 1, 2)))
with raises(ValueError, match="a let needs a body"):
    V.OfAny(term(E.let_("a", 1, None)))
with raises(ValueError, match="a quantifier needs a collection"):
    V.OfAny(term(E.quantifier("all", "x", None, True)))
assert FV.Interpreter(E.DIALECT, {"call": {}}, extension=lambda operator, thunks, node, scope: operator)(
    term(E.operation("dist", 1, 2))) == "dist"  # operators outside the vocabulary go to the extension

scope = SY.Variables({"x": 1})
x = term(E.variable("x"))
assert isinstance(scope, SY.Scope) and scope.lookup(x) == 1
inner = scope.bind("x", 2)
assert inner.lookup(x) == 2 and scope.lookup(x) == 1  # bind makes a new scope
with raises(KeyError, match="variable 'y' is not bound"):
    scope.lookup(term(E.variable("y")))
assert V.OfAny(x, inner) == 2  # an evaluator takes a scope as well as a mapping

<a id="FRM-09"></a>
## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
assert SY.free(term(basic), E.DIALECT) == {"person"} and SY.free(term(1), E.DIALECT) == set()
assert SY.free(term(E.let_("a", E.variable("a"), E.variable("a"))), E.DIALECT) == {"a"}  # the name is bound in the body only
assert SY.free(term(E.let_("a", None, E.variable("b"))), E.DIALECT) == {"b"}  # empty slots are skipped
assert SY.free(term(E.quantifier("all", "x", E.variable("xs"), E.variable("x").gt(E.variable("y")))), E.DIALECT) == {"xs", "y"}
assert SY.imports(term(basic), E.DIALECT) == []
assert FE.NameError is NameError and FE.ImportError is ImportError  # Python's own exceptions, by the framework's names
assert FE.OverflowError is OverflowError and FE.ZeroDivisionError is ZeroDivisionError and FE.IndexError is IndexError
assert SY.free(samples["Python3"], P.DIALECT) == {"person"}  # np, f and a are bound by imports and the let
assert SY.free(samples["TypeScript5"], TS.DIALECT) == {"person"}  # p by its arrow; properties and keys name, not refer
assert SY.free(term(P.call("hasattr", P.name("x"), "k"), P.DIALECT), P.DIALECT) == {"x"}  # builtins are ambient
assert SY.free(term(X.infix("+", X.cell("A1"), X.name("r")), X.DIALECT), X.DIALECT) == {"r"}  # cells are resolved by the workbook, not bound
assert SY.free(term(M.import_("pkg.*", M.identifier("x")), M.DIALECT), M.DIALECT) == {"x"}  # MATLAB's imports bind functions, not variables
assert [F.kind_of(i)[0] for i in SY.imports(samples["Python3"], P.DIALECT)] == ["import", "importfrom"]
assert [F.name_of(i) for i in SY.imports(samples["MATLAB"], M.DIALECT)] == ["pkg.*"]

<a id="FRM-10"></a>
## FRM-10 · A dialect may extend another: its kinds are the base's and its own, added to its union by schema extension

In [ ]:


def install(txn):
    """An extension of Patterns, added to its union: `twice(x)`, x + x; `import(module, body)`, an import binding the
    module's name within its body; `cell`, a reference the scope resolves without a binding; and `apply(function,
    arguments)`, a slot and variadic arguments. And `Bare`, a dialect of its own: its `Term`, `Expressions.Bare.Term`,
    the framework's literal and the import, the branch `bare` of Term."""
    S, Framework = txn.Schemas, txn.Expressions.Framework
    return txn.uid("frm.Ext/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Term": S.OfUnion, "Patterns": S.OfUnion, "Bare": S.OfUnion, "Twice": S.OfObject,
                  "Import": S.OfObject, "Cell": S.OfObject, "Apply": S.OfObject})
        .spec([
            lambda _: _.Twice.name("frm.Twice").properties({"operand": _.Term, "domain": S.OfStr}),
            lambda _: _.Import.name("frm.Import").properties({"name": S.OfStr, "body": _.Term, "domain": S.OfStr}),
            lambda _: _.Cell.name("frm.Cell").properties({"name": S.OfStr, "domain": S.OfStr}),
            lambda _: _.Apply.name("frm.Apply").properties({
                "function": _.Term, "arguments": S.OfContainer(lambda c: c.index([S.OfInt]).element([_.Term])), "domain": S.OfStr}),
            lambda _: _.Patterns.name("Expressions.Patterns.Expressions.Term").branches([  # Patterns' union, extended
                lambda t: t.name("twice").schema([_.Twice]), lambda t: t.name("import").schema([_.Import]),
                lambda t: t.name("cell").schema([_.Cell]), lambda t: t.name("apply").schema([_.Apply])]),
            lambda _: _.Bare.name("Expressions.Bare.Term").branches([
                lambda t: t.name("literal").schema([Framework.Literal]), lambda t: t.name("import").schema([_.Import])]),
            lambda _: _.Term.name("Expressions.Framework.Term").branches([lambda t: t.name("bare").schema([_.Bare])]),
        ])]))])


store.transaction(install)
Ext = F.Dialect("Ext", {
    "twice": F.Kind(F.APPLICATION, signature=D.Function((BD.Int,), BD.Int)),  # an open vocabulary: its tag names it
    "import": F.Kind(F.IMPORT, required=("name",), names=lambda attributes: (attributes["name"],)),
    "cell": F.Kind(F.REFERENCE, required=("name",), lexical=False),
    "apply": F.Kind(F.APPLICATION, signature=D.Opaque()),
}, domain_of=BD.of, extends=E.DIALECT)
assert list(Ext.kinds()) == [*E.DIALECT.kinds(), "twice", "import", "cell", "apply"] and Ext.base is E.DIALECT and E.DIALECT.base is None
assert (Ext.tag, Ext.union) == (E.DIALECT.tag, E.DIALECT.union)  # its terms are its base's union's
with raises(ValueError, match="an apply takes at least 1 arguments, got 0"):
    term(Ext.make(F.Form("apply")), Ext)
twice = lambda operand: E.Writer(lambda u: u.twice(lambda r: r.operand(Ext.spec(operand))))  # noqa: E731
import_ = lambda name, body, dialect=Ext: F.Writer(  # noqa: E731
    lambda u: getattr(u, "import")(lambda r: r.name(name).body(dialect.spec(body))), dialect.tag)
cell = lambda name: F.Writer(lambda u: u.cell(lambda r: r.name(name)), Ext.tag)  # noqa: E731
x = E.variable("x")
mixed = term(x.add(twice(x)), Ext)  # a writer of the base holds the extension's term
assert Ext.validate(mixed, bound={"x"}) == [] and F.kind_of(F.form(mixed).arguments[1])[0] == "twice"
assert E.DIALECT.validate(mixed, bound={"x"}) == ["argument 1: Patterns has no kind 'twice'"]  # the base reads only its own
with raises(ValueError, match="Patterns has no kind 'twice'"):
    V.OfAny(mixed, {"x": 1})


class Workbook(SY.Variables):
    """A scope with modules to import and cells to read."""

    def __init__(self, variables=None, cells=None, modules=None):
        super().__init__(variables)
        self.cells, self.modules = dict(cells or {}), dict(modules or {})

    def lookup(self, reference):
        if F.kind_of(reference)[0] == "cell":
            return self.cells[F.name_of(reference)]
        return super().lookup(reference)

    def enter(self, declaration):
        name = F.name_of(declaration)
        return self.bind(name, self.modules[name])


interpreter = FV.Interpreter(Ext, {"call": V.OPERATIONS, "quantifier": V.QUANTIFIERS,
                                   "twice": lambda thunks, node, scope: thunks[0]() * 2}, scope=Workbook)
assert interpreter(mixed, {"x": 3}) == 9
program = term(import_("m", E.variable("m").get("k").add(cell("A1"))), Ext)
assert interpreter(program, Workbook(cells={"A1": 1}, modules={"m": term(E.literal(1))})) is None  # no property k
assert Ext.validate(program) == [] and SY.free(program, Ext) == set()  # the import binds m; a cell needs no binding
assert [F.name_of(i) for i in SY.imports(program, Ext)] == ["m"]
assert Ext.infer(program).name() == "int | float" and Ext.infer(mixed, {"x": BD.Int}) is BD.Int
with raises(NotImplementedError, match="Variables has no imports"):
    SY.Variables().enter(program)
with raises(ValueError, match="an import needs a body"):
    interpreter(term(lambda u: getattr(u, "import")(lambda r: r.name("m")), Ext))
assert Ext.validate(term(lambda u: getattr(u, "import")(lambda r: r.body(E.variable("m"))), Ext)) == ["an import needs a name", "body: variable 'm' is not bound"]
# Partial evaluation of an extension's terms: an import is rebuilt, binding its names within its body.
reducer = FP.Reducer(interpreter, lambda value: E.literal(value) if type(value) is int else None, lambda node, results: None)
needs_y = term(import_("m", E.variable("m").get("k").add(E.variable("y"))), Ext)
assert F.same(term(reducer(needs_y, {"x": 1}), Ext), needs_y)
assert F.same(term(reducer(term(twice(x).add(twice(E.variable("y"))), Ext), {"x": 2}), Ext), term(E.literal(4).add(twice(E.variable("y"))), Ext))
# A dialect's kinds need no checks, domains or names of their own.
Bare = F.Dialect("Bare", {"literal": F.Kind(F.LITERAL), "import": F.Kind(F.IMPORT)}, domain_of=BD.of)
three = lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar(3)))  # noqa: E731: Bare has no writers
bare = term(import_("m", three, Bare), Bare)
assert Bare.validate(bare) == [] and SY.free(bare, Bare) == set() and Bare.infer(bare) is BD.Int
assert FV.Interpreter(Bare, {})(term(three, Bare)) == 3
with raises(ValueError, match="Clash would declare 'literal', which Patterns has"):
    F.Dialect("Clash", {"literal": F.Kind(F.APPLICATION)}, domain_of=BD.of, extends=E.DIALECT)
with raises(ValueError, match="Clash would declare 'error', which the framework has"):  # every dialect reads invalid syntax
    F.Dialect("Clash", {"error": F.Kind(F.APPLICATION)}, domain_of=BD.of)

<a id="FRM-11"></a>
## FRM-11 · Invalid syntax, `error` and `missing`: every dialect's `Term`, `Type` and `Pattern` hold them, validation reports them, evaluation refuses them by name, and printers write them as written

In [ ]:
from mbse.Expressions.C.Expressions import Text as CText
from mbse.Expressions.Cpp.Expressions import Text as CPPText
from mbse.Expressions.Excel.Expressions import Text as XText
from mbse.Expressions.LaTeX.Expressions import Text as LText
from mbse.Expressions.MATLAB.Expressions import Text as MText
from mbse.Expressions.Python.Python3 import Text as PText
from mbse.Expressions.SystemVerilog.Expressions import Text as SVText
from mbse.Expressions.TypeScript.TypeScript5 import Text as TSText

PRINTERS = {P.DIALECT: PText.ToText, M.DIALECT: MText.ToText, X.DIALECT: XText.ToText, L.DIALECT: LText.ToText,
            C.DIALECT: CText.ToText, CPP.DIALECT: CPPText.ToText, SV.DIALECT: SVText.ToText,
            TS.DIALECT: TSText.ToText}  # Patterns has none
assert all(write(holders[dialect]) == write(samples[dialect.name()]) for dialect, write in PRINTERS.items())  # FRM-04's
assert TSText.ToText(samples["TypeScript5"]) == (  # every kind, a body of statements written `{}`
    '({ a: [b,, ...c], d: { person }, f = 1 } = person), async function* g<T>(x: T = this.y) {}, async <U,>(p: U, .'
    '..r: U[]) => await import.meta.resolve(p), @person class K<T> extends person.Base<T> implements person {}, new'
    ' person.C<number>(...person), person`a${person}b`, `x${person}`, person++, typeof person, #p in person, person'
    ' ?? null, person ? /x/g : 1n, yield* person, person as string, person satisfies object, <any>person, person!, '
    'person<string>, [1,, ...person], { m(): void, ...person, [person]: "s" }, <div id="x" {...person}>{}{person}<>'
    'text</></div>, <><br /></>, person?.a, super.x, import(person)')


def recovered(dialect):
    """A name used as an expression, as each dialect's grammar makes it: TypeScript5's identifier, C++'s idexpression
    holding its identifier, the framework's variable."""
    if dialect is TS.DIALECT:
        return lambda v: v.identifier(lambda w: w.name("a"))
    if dialect is CPP.DIALECT:
        return lambda v: v.idexpression(lambda w: w.name(lambda n: n.identifier(lambda i: i.spelling("a"))))
    return lambda v: v.variable(lambda w: w.name("a"))


error = lambda dialect: lambda u: u.error(lambda r: r.text("(a +* 1)").terms([  # noqa: E731: what a parser skipped
    dialect.spec(recovered(dialect))]))  # and the terms it recovered, of its dialect
missing = lambda u: u.missing(lambda r: r.kind("expression"))  # noqa: E731: what it expected
for dialect in DIALECTS:
    skipped, expected = term(error(dialect), dialect), term(missing, dialect)
    assert F.form(skipped).kind == "error" and F.form(skipped).attributes == {"text": "(a +* 1)"}
    assert len(F.form(skipped).arguments) == 1 and F.form(expected) == F.Form("missing", {"kind": "expression"})
    assert dialect.validate(skipped) == ["invalid syntax '(a +* 1)'"] and dialect.validate(expected) == ["missing expression"]
    assert dialect.validate(term(lambda u: u.error(lambda r: r), dialect)) == ["invalid syntax"]
    assert dialect.validate(term(lambda u: u.missing(lambda r: r), dialect)) == ["missing a term"]
    assert F.same(term(dialect.make(F.form(skipped)), dialect), skipped)  # forms and make, as for every kind
    assert [dialect.kind(t)[2].role for t in (skipped, expected)] == [F.INVALID, F.INVALID]
    if dialect is SV.DIALECT:  # SystemVerilog's evaluator validates first, as it sizes its terms
        with raises(ValueError, match="cannot evaluate an invalid expression: invalid syntax '(a +* 1)'"):
            SVV.OfAny(skipped)
    elif dialect in EVALUATORS:
        with raises(ValueError, match=f"{dialect.name()} error is not evaluated"):
            EVALUATORS[dialect](skipped)
        with raises(ValueError, match=f"{dialect.name()} missing is not evaluated"):
            EVALUATORS[dialect](expected)
    if dialect in PRINTERS:  # an error as the source it spans, a missing term as nothing
        assert PRINTERS[dialect](skipped) == "(a +* 1)" if dialect is not X.DIALECT else XText.ToText(skipped) == "=(a +* 1)"
        assert PRINTERS[dialect](expected) == ("=" if dialect is X.DIALECT else "")
        with raises(ValueError, match=f"{dialect.name()} error has no text to print"):
            PRINTERS[dialect](term(lambda u: u.error(lambda r: r), dialect))
assert PText.ToText(term(P.binop("+", P.name("a"), missing), P.DIALECT)) == "a + "  # within a term, as written
assert SY.free(term(error(E.DIALECT)), E.DIALECT) == {"a"}  # the terms recovered within an error are terms
with raises(ValueError, match="Python3 error has no MATLAB counterpart"):  # nothing translates invalid syntax
    Translators.between(P.DIALECT, M.DIALECT).forward(term(error(P.DIALECT), P.DIALECT))
with raises(ValueError, match="cannot infer the domain of an invalid expression: missing expression"):
    E.DIALECT.infer(term(missing))
# Every dialect's Type and Pattern hold invalid syntax too: validation reports it within a type or a target, printers
# write it as written, evaluation refuses it by name
for union in ("Python.Python3.Pattern", "TypeScript.TypeScript5.Type", "TypeScript.TypeScript5.Pattern", "C.Expressions.Type",
              "C.Expressions.Pattern", "Cpp.Expressions.Type", "Cpp.Expressions.Pattern"):
    assert INVALID <= {b.name for b in store.registered(f"Expressions.{union}").branches}, union
one = lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar(1)))  # noqa: E731: Python3's and C's
named = term(lambda u: u.namedexpr(lambda r: r.target(lambda t: t.error(lambda e: e.text("1"))).value(one)), P.DIALECT)  # (1 := 1)
assert P.DIALECT.validate(named) == ["target: invalid syntax '1'"] and PText.ToText(named) == "(1 := 1)"
generator = term(P.generatorexp(P.name("p"), P.comprehension(lambda t: t.missing(lambda m: m.kind("identifier")), P.list_(1))), P.DIALECT)
assert P.DIALECT.validate(generator, {"p"}) == ["generators 0 target: missing identifier"] and PText.ToText(generator) == "(p for  in [1])"
with raises(ValueError, match="Python3 missing is not evaluated"):
    PV.OfAny(generator)
cast = term(lambda u: u.tsasexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.missing(lambda m: m.kind("type")))), TS.DIALECT)
assert TS.DIALECT.validate(cast, {"person"}) == ["missing type"] and TSText.ToText(cast) == "person as "
arrow = term(lambda u: u.arrowfunctionexpression(lambda r: r.params([lambda p: p.error(lambda e: e.text("a b"))])), TS.DIALECT)
assert TS.DIALECT.validate(arrow) == ["invalid syntax 'a b'"] and TSText.ToText(arrow) == "(a b) => {}"
for dialect, printer in [(C.DIALECT, CText.ToText), (CPP.DIALECT, CPPText.ToText)]:
    typed = term(lambda u: u.castexpression(lambda r: r.type(lambda t: t.error(lambda e: e.text("int int"))).operand(one)), dialect)
    assert dialect.validate(typed) == ["invalid syntax 'int int'"] and printer(typed) == "(int int)1"

<a id="FRM-12"></a>
## FRM-12 · A dialect reads only its own terms: a term holds its dialect, a shared kind's slot holds any dialect's term, and validation reports another's; C and C++ each define the kinds they share, each recording its standards

In [ ]:
foreign = term(P.let_("a", E.variable("a"), P.name("a")), P.DIALECT)  # a shared kind's slot: Patterns' variable in it
assert F.dialect_of(F.form(foreign).arguments[0]) == E.DIALECT.union and F.dialect_of(foreign) == P.DIALECT.union
assert P.DIALECT.validate(foreign) == ["value: a Patterns variable is not a Python3 term"]
with raises(ValueError, match="a Patterns variable is not a Python3 term"):
    PV.OfAny(foreign)
with raises(ValueError, match="an Excel call is not a MATLAB term"):
    MText.ToText(term(X.function("AND", True), X.DIALECT))
assert not F.same(term(E.variable("a")), term(P.name("a"), P.DIALECT))  # the same kind, in another dialect
own = term(P.binop("+", P.name("a"), 1), P.DIALECT)  # a dialect's own kind's slot: its union's terms alone
assert [F.dialect_of(a) for a in F.form(own).arguments] == [P.DIALECT.union] * 2
assert F.form(own).arguments[0].as_schema().name == P.DIALECT.union  # one level: its kind alone
with raises(Exception, match="'cell' is not a member of a union value"):  # the schema refuses a kind its union lacks
    term(P.binop("+", X.cell("A1"), 1), P.DIALECT)
# C and C++: each its own kinds and its own Term; each Kind records the standards of its language that have it, an
# extension none
assert C.STANDARDS == ("C11", "C17", "C23") and CPP.STANDARDS == ("C++17", "C++20", "C++23")
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):  # no literal of C++ holds bytes
    CPP.DIALECT.spec(b"x")
assert C.EXTENSIONS == CPP.EXTENSIONS == {"statementexpression", "extensionexpression"}  # GNU's
assert all(kind.standards == (() if tag in C.EXTENSIONS | INVALID else ("C23",) if tag == "constant" else C.STANDARDS)
           for tag, kind in C.DIALECT.kinds().items())  # nullptr: C23's
since_20 = {"requiresexpression", "awaitexpression", "yieldexpression", "designatedinitializer"}
assert all(kind.standards == (() if tag in CPP.EXTENSIONS | INVALID else ("C++20", "C++23") if tag in since_20 else CPP.STANDARDS)
           for tag, kind in CPP.DIALECT.kinds().items())
assert all(kind.standards == () for dialect in DIALECTS for tag, kind in dialect.kinds().items()
           if tag in INVALID or dialect not in (C.DIALECT, CPP.DIALECT, P.DIALECT, TS.DIALECT))
# TypeScript5: every kind is in every version of TypeScript 5; JSX, an extension, in none
assert TS.STANDARDS[0] == "TypeScript 5.0" and TS.STANDARDS[-1] == "TypeScript 5.9" and len(TS.STANDARDS) == 10
assert TS.EXTENSIONS == {"jsxelement", "jsxfragment", "jsxemptyexpression"}
assert all(kind.standards == (() if tag in TS.EXTENSIONS | INVALID else TS.STANDARDS) for tag, kind in TS.DIALECT.kinds().items())
# Python3: each kind records the versions of Python 3 that have it, from the one it arrived in; an extension, none
assert P.STANDARDS[0] == "Python 3.0" and P.STANDARDS[-1] == "Python 3.15" and len(P.STANDARDS) == 16
python3 = P.DIALECT.kinds()
assert python3[P.BINOP].standards == P.STANDARDS and python3[P.NAMEDEXPR].standards == P.STANDARDS[8:]
assert python3[P.TEMPLATESTR].standards == ("Python 3.14", "Python 3.15") and python3[P.AWAIT].standards[0] == "Python 3.5"
assert P.EXTENSIONS == {"let", "import", "importfrom"} and all(python3[tag].standards == () for tag in P.EXTENSIONS | INVALID)
assert CText.ToText(samples["C"]) == (  # every kind, a body of statements written `{}`
    '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, "a" "b", (person), _Generic(person, int: 1, d'
    'efault: 0), person++, sizeof(int), _Alignof(person), person += 1, .x[0] = person, (int[]){1}, ({}), __extension__ person}')
assert CPPText.ToText(samples["Cpp"]) == (
    '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, 10_km, "a" R"(b)", std::cout, this, (person), '
    '[=, &person](auto y) {}, (person + ...), requires (int z) { z + 1; }, int(person), person--, static_cast<int>(person'
    '), typeid(person), co_await person, sizeof(int), sizeof...(Ts), alignof(int), noexcept(person), new int{1}, delete['
    '] person, person = 0, throw, co_yield person, person..., .x = person, ({}), __extension__ person}')
assert CV.OfAny(term(C.binary("+", 1, 2), C.DIALECT)) == CPV.OfAny(term(CPP.binary("+", 1, 2), CPP.DIALECT)) == 3
with raises(Exception, match="'call' is not a member of a union value"):  # C's own kinds hold C's terms alone
    term(C.binary("+", E.variable("x").add(1), 1), C.DIALECT)